In [ ]:
import os
from dotenv import load_dotenv

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import CodeInterpreterTool
from azure.identity import DefaultAzureCredential
from typing import Any
from pathlib import Path
from datetime import datetime



In [ ]:
load_dotenv()
project_client = AIProjectClient.from_connection_string(
    credential=DefaultAzureCredential(), conn_str=os.environ["PROJECT_CONNECTION_STRING"]
)

In [ ]:
from IPython.display import display, HTML, Image
from pathlib import Path


async def run_agent_with_visualization():
    html_output = "<h2>Azure AI 에이전트 실행</h2>"

    with project_client:
        # CodeInterpreterTool 인스턴스 생성
        code_interpreter = CodeInterpreterTool()

        # CodeInterpreterTool은 에이전트 생성 시 포함되어야 함
        # 사용 사례에 맞게 Azure AI Foundry에 배포된 올바른 모델 이름을 설정해야 함
        agent = project_client.agents.create_agent(
            model="gpt-4o-mini",
            name="my-agent",
            instructions="You are helpful agent",
            tools=code_interpreter.definitions,
            tool_resources=code_interpreter.resources,
        )
        html_output += f"<div><strong>에이전트 생성됨</strong> ID: {agent.id}</div>"

        # 스레드 생성
        thread = project_client.agents.create_thread()
        html_output += f"<div><strong>스레드 생성됨</strong> ID: {thread.id}</div>"

        # 사용자 쿼리 - 보기 좋게 표시
        user_query = "Could you please create a bar chart for the operating profit using the following data and provide the file to me? Bali: 100 Travelers, Paris: 356 Travelers, London: 900 Travelers, Tokyo: 850 Travellers"
        html_output += "<div style='margin:15px 0; padding:10px; background-color:#f5f5f5; border-left:4px solid #007bff; border-radius:4px;'>"
        html_output += "<strong>사용자:</strong><br>"
        html_output += f"<div style='margin-left:15px'>{user_query}</div>"
        html_output += "</div>"

        # 메시지 생성
        message = project_client.agents.create_message(
            thread_id=thread.id,
            role="user",
            content=user_query,
        )

        # 에이전트 실행 - "처리 중" 메시지 표시
        display(HTML(
            html_output + "<div style='color:#007bff'><i>요청 처리 중...</i></div>"))

        # 실행
        run = project_client.agents.create_and_process_run(
            thread_id=thread.id, agent_id=agent.id)

        # 상태 업데이트
        status_color = 'green' if run.status == 'completed' else 'red'
        html_output += f"<div><strong>실행 완료</strong> 상태: <span style='color:{status_color}'>{run.status}</span></div>"

        if run.status == "failed":
            html_output += f"<div style='color:red'><strong>실행 실패:</strong> {run.last_error}</div>"

        # 스레드에서 메시지 가져오기
        messages = project_client.agents.list_messages(thread_id=thread.id)

        # 어시스턴트 응답 포맷
        html_output += "<div style='margin:15px 0; padding:10px; background-color:#f0f7ff; border-left:4px solid #28a745; border-radius:4px;'>"
        html_output += "<strong>어시스턴트:</strong><br>"

        # 실제 구조를 기반으로 메시지 처리
        # 먼저 어시스턴트의 텍스트 응답 가져오기 시도
        try:
            # 첫 번째 접근 - messages가 role 속성을 가진 객체의 리스트인 경우
            assistant_msgs = [msg for msg in messages if hasattr(
                msg, 'role') and msg.role == "assistant"]

            if assistant_msgs:
                last_msg = assistant_msgs[-1]
                if hasattr(last_msg, 'content'):
                    if isinstance(last_msg.content, list):
                        for content_item in last_msg.content:
                            if hasattr(content_item, 'type') and content_item.type == "text":
                                html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{content_item.text.value}</div>"
                    elif isinstance(last_msg.content, str):
                        html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{last_msg.content}</div>"

            # 위 접근 방식으로 메시지를 찾지 못한 경우, 다른 구조 시도
            if not assistant_msgs:
                # messages가 속성을 가진 클래스인 경우
                if hasattr(messages, 'data'):
                    for msg in messages.data:
                        if hasattr(msg, 'role') and msg.role == "assistant":
                            if hasattr(msg, 'content'):
                                html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{msg.content}</div>"

        except Exception as e:
            html_output += f"<div style='color:red'><strong>메시지 처리 오류:</strong> {str(e)}</div>"

        html_output += "</div>"

        # 실제 구조를 기반으로 이미지 콘텐츠 처리
        saved_images = []
        try:
            # image_contents를 속성으로 접근 시도
            if hasattr(messages, 'image_contents'):
                for image_content in messages.image_contents:
                    file_id = image_content.image_file.file_id
                    file_name = f"{file_id}_image_file.png"
                    project_client.agents.save_file(
                        file_id=file_id, file_name=file_name)
                    saved_images.append(file_name)
                    html_output += f"<div style='margin-top:10px'><strong>생성된 이미지:</strong> {file_name}</div>"
        except Exception as e:
            html_output += f"<div style='color:orange'><i>참고: 이미지를 찾을 수 없거나 이미지 처리 중 오류 발생</i></div>"

        # 실제 구조를 기반으로 파일 경로 주석 처리
        try:
            # file_path_annotations를 속성으로 접근 시도
            if hasattr(messages, 'file_path_annotations'):
                for file_path_annotation in messages.file_path_annotations:
                    file_name = Path(file_path_annotation.text).name
                    project_client.agents.save_file(
                        file_id=file_path_annotation.file_path.file_id, file_name=file_name)
                    html_output += "<div style='margin:10px 0; padding:8px; background-color:#f8f9fa; border:1px solid #ddd; border-radius:4px;'>"
                    html_output += f"<strong>생성된 파일:</strong> {file_name}<br>"
                    html_output += f"<strong>유형:</strong> {file_path_annotation.type}<br>"
                    html_output += "</div>"
        except Exception as e:
            html_output += f"<div style='color:orange'><i>참고: 파일 주석을 찾을 수 없거나 파일 처리 중 오류 발생</i></div>"

        # 완료 후 에이전트 삭제
        project_client.agents.delete_agent(agent.id)
        html_output += "<div style='margin-top:10px'><i>완료 후 에이전트 삭제됨</i></div>"

        # 모든 콘텐츠의 최종 표시
        display(HTML(html_output))

        # 저장된 이미지 표시
        for img_file in saved_images:
            display(Image(img_file))

# 함수 실행
await run_agent_with_visualization()